<a href="https://colab.research.google.com/github/SlukaLukas2/Neuronov-s-t-a-evolu-n-soustavy-NSES-AI---artifical-intelligence/blob/main/NSES_09_Tokenizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Requirements: Upload the file NSES_1_autocorrect.txt in your session


In [ ]:
!ls
!pwd

NSES_1_latex.txt  sample_data
/content


In [ ]:
import json, os
from typing import Dict, List, Tuple

In [ ]:
# We always start with a dataset to train on. Let's upload our first lecture
# read it in to inspect it
with open('NSES_1_latex.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [ ]:
print("length of dataset in characters: ", len(text))
print(type(text))

length of dataset in characters:  110265
<class 'str'>


In [ ]:
# let's look at some defined range of characters
indices_for_testing = slice(928, 2000)
text_for_testing = text[indices_for_testing]
print(text[indices_for_testing])


Pač původně to byl nějakej, museli jsme náchodit nějakou neuronku, napsat k tomu nějakej report a když jsem si tak počítal, kolik by mi tak trvalo vám všechny semestrálky opravit, tak jsem se v tom určitém případě dostal na 2 týdny 8 hodinové práce a v tom horším případě až na 5 týdnů.
Skutečně bohužel nemám 5 týdnů času na to, abych opravil všechny semestrálky.
Možná si říkáte, proč vždyť je to moje práce.
Já jsem především teda vědec, takže zkoumám, píšu články a to učení samozřejmě mám rád, ale je to velice, velice vlastně jenom drobný podíl toho, co tady sady děláme.
Bývá to tak docela běžné na na katedře u nás na kybernetice, na jiných katedrách se víc specializujou třeba na to učení.
Pak tedy Vlasta Radová docentka je garantem toho předmětu.
Ta se zaměřuje na evoluční strategie, které budete mít na konci semestru.
Pak jsem já, Marek Hrúz. Samozřejmě sta čárka na mým indikuje, že jsem úvodem Slovák, proto budu občas mluvit, takže si budete říkat, jestli je to ukrajinec nebo něco,

In [ ]:
# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
print("Unique characters: " + ''.join(chars))
print("Vocabulary size: " + str(vocab_size))

Unique characters: 
 $%'()+,-.0123456789=?ABCDEFGHIJKLMNOPRSTUVWXYZ\^_abcdefghijklmnopqrstuvwxyz{}ÁÚáéíóúýČčďěňŘřŠšťůŽž
Vocabulary size: 100


1) Very simple approach to tokenizing characters
- for sorted list of all possible characters - we assign it an integer
- reversly - decoding- we create a dictionary which will to each integer assign a charachter

In [ ]:
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

print(encode(text_for_testing))
print(decode(encode(text_for_testing)))

[0, 38, 51, 88, 1, 66, 97, 72, 65, 54, 64, 90, 1, 70, 65, 1, 52, 75, 62, 1, 64, 90, 60, 51, 61, 55, 60, 8, 1, 63, 71, 69, 55, 62, 59, 1, 60, 69, 63, 55, 1, 64, 81, 53, 58, 65, 54, 59, 70, 1, 64, 90, 60, 51, 61, 65, 71, 1, 64, 55, 71, 68, 65, 64, 61, 71, 8, 1, 64, 51, 66, 69, 51, 70, 1, 61, 1, 70, 65, 63, 71, 1, 64, 90, 60, 51, 61, 55, 60, 1, 68, 55, 66, 65, 68, 70, 1, 51, 1, 61, 54, 75, 99, 1, 60, 69, 55, 63, 1, 69, 59, 1, 70, 51, 61, 1, 66, 65, 88, 83, 70, 51, 62, 8, 1, 61, 65, 62, 59, 61, 1, 52, 75, 1, 63, 59, 1, 70, 51, 61, 1, 70, 68, 72, 51, 62, 65, 1, 72, 81, 63, 1, 72, 95, 55, 53, 58, 64, 75, 1, 69, 55, 63, 55, 69, 70, 68, 81, 62, 61, 75, 1, 65, 66, 68, 51, 72, 59, 70, 8, 1, 70, 51, 61, 1, 60, 69, 55, 63, 1, 69, 55, 1, 72, 1, 70, 65, 63, 1, 71, 68, 88, 59, 70, 82, 63, 1, 66, 93, 83, 66, 51, 54, 90, 1, 54, 65, 69, 70, 51, 62, 1, 64, 51, 1, 13, 1, 70, 86, 54, 64, 75, 1, 19, 1, 58, 65, 54, 59, 64, 65, 72, 82, 1, 66, 68, 81, 53, 55, 1, 51, 1, 72, 1, 70, 65, 63, 1, 58, 65, 68, 95, 83,

2) More complicated - Byte Pair Encoding
- Start with one token per character
- Merge common pairs of tokens into a token
- Repeat until desired vocab size or all merged

In [ ]:
tokens = text.encode('utf-8') # raw bytes -
print(tokens[indices_for_testing])
tokens = list(map(int, tokens)) # convert to a list of integers in range 0..255 for convenience
print('---')
print(text[indices_for_testing])
print("length:", len(text))
print('---')
print(tokens[indices_for_testing])
print("length:", len(tokens))
print("max value:", max(tokens))

b'uset tro\xc5\xa1ku p\xc5\x99izp\xc5\xafsobit semestr\xc3\xa1ln\xc3\xad pr\xc3\xa1ce, abysme to v\xc5\xa1echno n\xc4\x9bjak rozumn\xc4\x9b zvl\xc3\xa1dli.\nPa\xc4\x8d p\xc5\xafvodn\xc4\x9b to byl n\xc4\x9bjakej, museli jsme n\xc3\xa1chodit n\xc4\x9bjakou neuronku, napsat k tomu n\xc4\x9bjakej report a kdy\xc5\xbe jsem si tak po\xc4\x8d\xc3\xadtal, kolik by mi tak trvalo v\xc3\xa1m v\xc5\xa1echny semestr\xc3\xa1lky opravit, tak jsem se v tom ur\xc4\x8dit\xc3\xa9m p\xc5\x99\xc3\xadpad\xc4\x9b dostal na 2 t\xc3\xbddny 8 hodinov\xc3\xa9 pr\xc3\xa1ce a v tom hor\xc5\xa1\xc3\xadm p\xc5\x99\xc3\xadpad\xc4\x9b a\xc5\xbe na 5 t\xc3\xbddn\xc5\xaf.\nSkute\xc4\x8dn\xc4\x9b bohu\xc5\xbeel nem\xc3\xa1m 5 t\xc3\xbddn\xc5\xaf \xc4\x8dasu na to, abych opravil v\xc5\xa1echny semestr\xc3\xa1lky.\nMo\xc5\xben\xc3\xa1 si \xc5\x99\xc3\xadk\xc3\xa1te, pro\xc4\x8d v\xc5\xbedy\xc5\xa5 je to moje pr\xc3\xa1ce.\nJ\xc3\xa1 jsem p\xc5\x99edev\xc5\xa1\xc3\xadm teda v\xc4\x9bdec, tak\xc5\xbee zkoum\xc3\xa1m, p\xc3\

In [ ]:
# we have to continuesly getting statistics about
# charactes and their occurencies
def get_stats(ids: List[int]) -> Dict[Tuple[int, int], int]:
    counts = {}
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts

# here we simply merge ids into one pair
def merge(ids: List[int], pair: Tuple[int, int], idx: int) -> List[int]:
    newids = []
    i = 0
    a, b = pair
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == a and ids[i+1] == b:
            newids.append(idx)
            i += 2
        else:
            newids.append(ids[i])
            i += 1
    return newids

def tok_to_str(t: int | tuple[int, ...], base_vocab_size: int = 256) -> str:
    """
    Converts a token (int or tuple of ints) back into a human-readable string.
    Handles multi-byte UTF-8 characters (e.g., č, š, ř).
    """
    # If it's a merged token id (not raw bytes)
    if isinstance(t, int) and t >= base_vocab_size:
        return f"<{t}>"

    # If it's a single byte
    if isinstance(t, int):
        b = bytes([t])
    # Or a tuple/list of bytes
    else:
        b = bytes(t)

    try:
        s = b.decode('utf-8')
        # Only show printable characters; else show hex escapes
        return s if s.isprintable() else ''.join(f"\\x{x:02x}" for x in b)
    except UnicodeDecodeError:
        return ''.join(f"\\x{x:02x}" for x in b)


In [ ]:
def train_bpe_from_text(text: str, num_merges: int, base_vocab_size: int = 256):
    ids = list(text.encode("utf-8"))
    merges_ordered, merges_dict, stats = [], {}, {}

    # initialize vocab: each base token is one byte
    vocab = {i: bytes([i]) for i in range(base_vocab_size)}
    next_id = base_vocab_size

    for _ in range(num_merges):
        stats = get_stats(ids)
        if not stats:
            break

        pair = max(stats, key=stats.get)
        new_id = next_id
        next_id += 1

        ids = merge(ids, pair, new_id)

        # build the new token’s byte sequence
        b1, b2 = pair
        vocab[new_id] = vocab[b1] + vocab[b2]

        # decode the merged bytes for printing
        try:
            chars = vocab[new_id].decode('utf-8')
            if not chars.isprintable():
                chars = ''.join(f"\\x{x:02x}" for x in vocab[new_id])
        except UnicodeDecodeError:
            chars = ''.join(f"\\x{x:02x}" for x in vocab[new_id])

        count = stats[pair]
        print(f"merging {pair}, characters: {chars}, occurred {count} → new token {new_id}")

        merges_ordered.append((pair, new_id))
        merges_dict[pair] = new_id

    return stats, merges_ordered, merges_dict, vocab

In [ ]:
Pair = Tuple[int, int]

class BPETokenizer:
    """
    Byte-level BPE tokenizer that relies on the existing helpers:
      - get_stats(ids)
      - merge(ids, pair, new_id)
      - train_bpe_from_text(text, num_merges, base_vocab_size)
    """

    def __init__(self, base_vocab_size: int = 256):
        self.base_vocab_size = base_vocab_size
        self.vocab: Dict[int, bytes] = {i: bytes([i]) for i in range(base_vocab_size)}
        self.merges_ordered: List[Tuple[Pair, int]] = []
        self.merges_dict: Dict[Pair, int] = {}
        self.rank: Dict[Pair, int] = {}  # lower = higher priority

    # ---- training ----
    def train(self, text: str, num_merges: int):
        """
        Learns merges & vocab from `text` using your provided training function.
        """
        _, merges_ordered, merges_dict, vocab = train_bpe_from_text(
            text=text, num_merges=num_merges, base_vocab_size=self.base_vocab_size
        )
        self.merges_ordered = merges_ordered
        self.merges_dict = merges_dict
        self.vocab.update(vocab)
        self.rank = {pair: i for i, (pair, _) in enumerate(self.merges_ordered)}

    # ---- core BPE apply ----
    def _apply_bpe(self, ids: List[int]) -> List[int]:
        """
        Greedily applies learned merges by repeatedly choosing the present pair
        with the best (lowest) rank, then merging all its non-overlapping occurrences.
        """
        if not self.rank:
            return ids[:]
        cur = ids[:]
        while True:
            stats = get_stats(cur)
            candidates = [(self.rank[p], p) for p in stats if p in self.rank]
            if not candidates:
                break
            _, best_pair = min(candidates, key=lambda x: x[0])
            new_id = self.merges_dict[best_pair]
            cur = merge(cur, best_pair, new_id)
        return cur

    # ---- public API ----
    def encode(self, text: str) -> List[int]:
        """
        UTF-8 -> byte ids -> apply BPE merges.
        """
        byte_ids = list(text.encode("utf-8"))
        return self._apply_bpe(byte_ids)

    def encode_bytes(self, data: bytes) -> List[int]:
        """
        Directly encode raw bytes (skips UTF-8 step).
        """
        return self._apply_bpe(list(data))

    def decode(self, ids: List[int]) -> str:
        """
        Token ids -> bytes -> UTF-8 string (errors='replace' for safety).
        """
        buf = bytearray()
        for tid in ids:
            if tid not in self.vocab:
                raise ValueError(f"Unknown token id: {tid}")
            buf.extend(self.vocab[tid])
        return buf.decode("utf-8", errors="replace")

    # ---- (optional) persistence helpers ----
    def state_dict(self) -> dict:
        return {
            "base_vocab_size": self.base_vocab_size,
            "vocab": {int(k): v.hex() for k, v in self.vocab.items()},
            "merges_ordered": [(pair, int(new_id)) for pair, new_id in self.merges_ordered],
        }

    @classmethod
    def from_state_dict(cls, state: dict) -> "BPETokenizer":
        tok = cls(base_vocab_size=state["base_vocab_size"])
        tok.vocab = {int(k): bytes.fromhex(v) for k, v in state["vocab"].items()}
        tok.merges_ordered = [(tuple(pair), int(new_id)) for pair, new_id in state["merges_ordered"]]
        tok.merges_dict = {pair: new_id for pair, new_id in tok.merges_ordered}
        tok.rank = {pair: i for i, (pair, _) in enumerate(tok.merges_ordered)}
        return tok


In [ ]:
# --- Example usage of BPETokenizer - small vocab size ---
## our vocab size is not max(tokens)
vocab_size = max(tokens) + 6 # the desired final vocabulary size
# increase teh vocab size - will lead to decrease of context lenght
num_merges = vocab_size - max(tokens)
#copy_tokens = list(tokens) # copy so we don't destroy the original list

# 1. Create the tokenizer
tokenizer = BPETokenizer(base_vocab_size=256)

# 2. Train on some text (learns merges and vocab)
tokenizer.train(text=text, num_merges=num_merges)

# 3. Encode text → token IDs
encoded = tokenizer.encode(text)
print("Encoded IDs:", encoded[indices_for_testing])

# 4. Decode token IDs → text
decoded = tokenizer.decode(encoded)
print("Decoded text:", decoded[indices_for_testing])

# 5. (Optional) Save and reload tokenizer state
state = tokenizer.state_dict()
reloaded = BPETokenizer.from_state_dict(state)

# 6. Verify reloaded tokenizer works the same
encoded2 = reloaded.encode(text)
print("Reloaded encoded:", encoded2[indices_for_testing])

print("Original tokens length:", len(tokens))
print("BPE tokens length:", len(encoded))
print(f"Compression ratio: {len(tokens) / len(encoded):.2f}X")

merging (44, 32), characters: , , occurred 3965 → new token 256
merging (101, 32), characters: e , occurred 2817 → new token 257
merging (196, 155), characters: ě, occurred 1998 → new token 258
merging (32, 116), characters:  t, occurred 1775 → new token 259
merging (195, 173), characters: í, occurred 1752 → new token 260
merging (195, 161), characters: á, occurred 1745 → new token 261
Encoded IDs: [98, 121, 108, 32, 110, 258, 106, 97, 107, 101, 106, 256, 109, 117, 115, 101, 108, 105, 32, 106, 115, 109, 257, 110, 261, 99, 104, 111, 100, 105, 116, 32, 110, 258, 106, 97, 107, 111, 117, 32, 110, 101, 117, 114, 111, 110, 107, 117, 256, 110, 97, 112, 115, 97, 116, 32, 107, 259, 111, 109, 117, 32, 110, 258, 106, 97, 107, 101, 106, 32, 114, 101, 112, 111, 114, 116, 32, 97, 32, 107, 100, 121, 197, 190, 32, 106, 115, 101, 109, 32, 115, 105, 259, 97, 107, 32, 112, 111, 196, 141, 260, 116, 97, 108, 256, 107, 111, 108, 105, 107, 32, 98, 121, 32, 109, 105, 259, 97, 107, 259, 114, 118, 97, 108, 111,

In [ ]:
# --- Example usage of BPETokenizer - big vocab size ---
## our vocab size is not max(tokens)
vocab_size = max(tokens) + 1000 # the desired final vocabulary size
# increase teh vocab size - will lead to decrease of context lenght
num_merges = vocab_size - max(tokens)
copy_tokens = list(tokens) # copy so we don't destroy the original list

# 1. Create the tokenizer
tokenizer = BPETokenizer(base_vocab_size=256)

# 2. Train on some text (learns merges and vocab)
tokenizer.train(text=text, num_merges=num_merges)

# 3. Encode text → token IDs
encoded = tokenizer.encode(text)
print("Encoded IDs:", encoded[indices_for_testing])

# 4. Decode token IDs → text
decoded = tokenizer.decode(encoded)
print("Decoded text:", decoded[indices_for_testing])

# 5. (Optional) Save and reload tokenizer state
state = tokenizer.state_dict()
reloaded = BPETokenizer.from_state_dict(state)

# 6. Verify reloaded tokenizer works the same
encoded2 = reloaded.encode(text)
print("Reloaded encoded:", encoded2[indices_for_testing])

print("Original tokens length:", len(tokens))
print("BPE tokens length:", len(encoded))
print(f"Compression ratio: {len(tokens) / len(encoded):.2f}X")

merging (44, 32), characters: , , occurred 3965 → new token 256
merging (101, 32), characters: e , occurred 2817 → new token 257
merging (196, 155), characters: ě, occurred 1998 → new token 258
merging (32, 116), characters:  t, occurred 1775 → new token 259
merging (195, 173), characters: í, occurred 1752 → new token 260
merging (195, 161), characters: á, occurred 1745 → new token 261
merging (97, 107), characters: ak, occurred 1669 → new token 262
merging (111, 32), characters: o , occurred 1506 → new token 263
merging (197, 190), characters: ž, occurred 1416 → new token 264
merging (115, 116), characters: st, occurred 1082 → new token 265
merging (110, 258), characters: ně, occurred 904 → new token 266
merging (32, 106), characters:  j, occurred 879 → new token 267
merging (114, 111), characters: ro, occurred 842 → new token 268
merging (32, 115), characters:  s, occurred 828 → new token 269
merging (101, 110), characters: en, occurred 811 → new token 270
merging (32, 118), characte

In [ ]:
# ---- Convert merges (list or dict) to a parent map: new_id -> (a, b)
def to_parent_map(merges):
    """
    Accepts:
      - merges_dict: { (a,b): new_id }
      - merges_ordered: [ ((a,b), new_id), ... ]
    Returns:
      parents: { new_id: (a, b) }
    """
    parents = {}
    if isinstance(merges, dict):
        for (a, b), new_id in merges.items():
            parents[new_id] = (a, b)
    elif isinstance(merges, list):
        for pair, new_id in merges:
            a, b = pair
            parents[new_id] = (a, b)
    else:
        raise TypeError("`merges` must be a dict {(a,b): new_id} or a list [((a,b), new_id), ...].")
    return parents


# ---------- reconstruct merged token strings ----------
def build_string_from_parents(parents, index, cache=None, base_vocab_size: int = 256):
    """
    Reconstruct the *byte-char* string for a token id using the parent map.
    - If index < base_vocab_size: it's a single byte -> chr(index)
    - Else if index in parents: recursively build left + right
    Returns a latin-1 string (one char per byte).
    """
    if cache is None:
        cache = {}

    if index in cache:
        return cache[index]

    if index < base_vocab_size:
        s = chr(index)
        cache[index] = s
        return s

    if index in parents:
        a, b = parents[index]
        left  = build_string_from_parents(parents, a, cache, base_vocab_size)
        right = build_string_from_parents(parents, b, cache, base_vocab_size)
        s = left + right
        cache[index] = s
        return s

    # Not found (shouldn't happen if merges are complete)
    cache[index] = ''
    return ''


def get_all_merged_strings(merges, decode_utf8: bool = True, base_vocab_size: int = 256):
    """
    Accepts either merges_dict or merges_ordered.
    Returns:
      { new_id: printable_string }
    """
    parents = to_parent_map(merges)
    cache = {}
    results = {}
    for new_id in sorted(parents.keys()):
        byte_chars = build_string_from_parents(parents, new_id, cache, base_vocab_size)
        if decode_utf8:
            printable = byte_chars.encode('latin-1').decode('utf-8', errors='replace')
        else:
            printable = byte_chars
        results[new_id] = printable
    return results



# Works with either structure:
merged_strings = get_all_merged_strings(tokenizer.merges_ordered, decode_utf8=True)
# or:
# merged_strings = get_all_merged_strings(merges_dict, decode_utf8=True)

for new_id, string in merged_strings.items():
    if len(string) > 6:
        print(f"Token {new_id} -> '{string}'")


Token 385 -> ' vlastně'
Token 391 -> ', že jo'
Token 407 -> 'amozřej'
Token 408 -> 'amozřejmě'
Token 418 -> ' neuron'
Token 482 -> ', které'
Token 484 -> ' nějaké'
Token 533 -> ', takže '
Token 553 -> ' prostě'
Token 555 -> ', nějak'
Token 556 -> ' samozřejmě'
Token 562 -> 'e nějak'
Token 580 -> 'teligen'
Token 590 -> 'znamená'
Token 593 -> 'e vlastně'
Token 596 -> ', protože '
Token 605 -> ', jestli'
Token 606 -> ', který'
Token 612 -> ', je to '
Token 613 -> 'vlastně'
Token 615 -> '.
Takže '
Token 619 -> ' je to '
Token 631 -> ', nebo '
Token 660 -> ' tak dále'
Token 666 -> ' výstup'
Token 672 -> ' inteligen'
Token 685 -> ', jako '
Token 692 -> ' trošku'
Token 695 -> ', prostě'
Token 700 -> ' jednič'
Token 710 -> 'aktivační'
Token 745 -> ' nějakou'
Token 747 -> ' systém'
Token 751 -> ', jakoby'
Token 752 -> 'aktivační funk'
Token 753 -> 'arametr'
Token 761 -> ' se to '
Token 763 -> ', která'
Token 766 -> ' takhle '
Token 782 -> ' způsob'
Token 783 -> 'ové sítě'
Token 799 -> ' takové'